# NOT training with smoothing

In [ ]:
import os, sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import clear_output

device = "cuda" if torch.cuda.is_available() else "cpu"

from src.utils import cost, show_mapping , grad_norm, cosine_lr, sigma_schedule, plot_3d_function

## Distributions

In [ ]:
# source distribution μ
def sample_mu(batch_size, device="cpu"):
    x = 2 * torch.rand(batch_size, 1, device=device) - 1  # [-1,1]
    y = torch.zeros(batch_size, 1, device=device)         # y = 0
    return torch.cat([x, y], dim=1)


# target distribution ν
def sample_nu(batch_size, device="cpu"):
    x = torch.zeros(batch_size, 1, device=device)         # y = 0
    y = 2 * torch.rand(batch_size, 1, device=device) - 1  # [-1,1]
    return torch.cat([x, y], dim=1)

In [ ]:
def sample_mu(batch_size, device="cpu", n_grid=5):
    grid = torch.linspace(-1, 1, 2*n_grid-1, device=device)  # grid points
    idx = 2*torch.randint(1, n_grid, (batch_size,), device=device)-1

    y = grid[idx].unsqueeze(1)
    x = 2 * torch.rand(batch_size, 1, device=device) - 1

    return torch.cat([x, y], dim=1)

def sample_nu(batch_size, device="cpu", n_grid=5):
    grid = torch.linspace(-1, 1, 2*n_grid-1, device=device)  # grid points
    idx = 2*torch.randint(1, n_grid, (batch_size,), device=device)-1

    x = grid[idx].unsqueeze(1)
    y = 2 * torch.rand(batch_size, 1, device=device) - 1

    return torch.cat([x, y], dim=1)

In [ ]:
def get_hidden_dim(d):
    if d in [2, 4]:
        return 256
    else:
        return 1024


# Potential function v_phi: R^d -> R
class Critic(nn.Module):
    def __init__(self, d):
        super().__init__()
        h = get_hidden_dim(d)

        self.net = nn.Sequential(
            nn.Linear(d, h),
            nn.ReLU(),
            nn.Linear(h, 1)
        )

    def forward(self, x):
        return self.net(x)


# Transport map T_theta: R^d -> R^d
class Transport(nn.Module):
    def __init__(self, d):
        super().__init__()
        h = get_hidden_dim(d)

        self.net = nn.Sequential(
            nn.Linear(d, h),
            nn.ReLU(),
            nn.Linear(h, d)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
d = 2

T = Transport(d).to(device)
f = Critic(d).to(device)

## GDmax Training

In [ ]:
lr_T = 1e-4
lr_f = 1e-4

K = 20

optimizer_T = torch.optim.Adam(T.parameters(), lr=lr_T)
optimizer_f = torch.optim.Adam(f.parameters(), lr=lr_f, maximize= True)

batchsize_x = 128
batchsize_y = 128
sigma = 0.2

for step in range(20000):
    x = sample_mu(batchsize_x).to(device)
    y = sample_nu(batchsize_y).to(device)
    z = torch.randn(batchsize_x, d, device=device)

    x = x + sigma * z

    for _ in range(K):
        Tx = T(x)
        loss = cost(x, Tx).mean() - f(Tx).mean() + f(y).mean()

        optimizer_T.zero_grad()
        loss.backward()
        optimizer_T.step()

    Tx = T(x)
    loss = cost(x, Tx).mean() - f(Tx).mean() + f(y).mean()

    optimizer_f.zero_grad()
    loss.backward()
    optimizer_f.step()

    Tx = T(x)
    loss = cost(x, Tx).mean() - f(Tx).mean() + f(y).mean()

    T_grads = torch.autograd.grad(loss, T.parameters(), retain_graph=True)
    f_grads = torch.autograd.grad(loss, f.parameters())


    if (step+1) % 500 == 0:
        sigma = sigma_schedule(step+1, 20000)
        clear_output(wait=True)
        print(f"step {step+1}, loss {loss.item():.4f}")
        show_mapping(T, sample_mu, sample_nu)

## Plot Results

In [ ]:
show_mapping(T,sample_mu,sample_nu, option = True)

In [ ]:
option = True

x = sample_mu(1000).to(device)
z = torch.randn(1000, d, device=device)

x_tilde = x + sigma * z

outputs = []
for _ in range(5):
    outputs.append(T(x_tilde).detach().cpu())

outputs = torch.cat(outputs, dim=0)
x = x.cpu()
y = sample_nu(1000).cpu()


plt.figure()

if (option == True):
    for i in range(50):
        plt.arrow(x[i,0], x[i,1],
              outputs[i,0]-x[i,0],
              outputs[i,1]-x[i,1],
              color='gray', alpha=0.5, head_width=0, length_includes_head=True)

plt.scatter(x[:,0], x[:,1], label=r"$x \sim \mu$", s=1, alpha=0.5)
plt.scatter(y[:,0], y[:,1], label=r"$y \sim \nu$", s=1, alpha=0.5)
plt.scatter(outputs[:,0], outputs[:,1], label="T(x+z)", s=1, alpha=0.5)
plt.axis('equal')
plt.legend()
plt.show()

In [ ]:
plot_3d_function(f)